# Portfolio Risk Management: Factor Models, Stress Testing and Crisis Hedging

This notebook covers the full workflow of:
* factor-model-driven portfolio construction
* stress testing under two geopolitical crises
* risk-based rebalancing

**Two crisis tracks:**
- Gulf Crisis: oil spike, stagflation, EM stress
- Ukraine/Russia: energy and food shock, defense surge, European risk premium

**Workflows:**
1. BUILD: Build factor model and benign portfolio
2. BUILD: Define crisis portfolios for each scenario
3. RISK: Stress test benign portfolio against both crises
4. RISK: Hedge and rebalance
5. ANALYSIS: Compare outcomes

---
## Part 1: Factor Model Foundations

### 1.1 Basics

A factor model decomposes portfolio returns into:
- **Systematic returns**: driven by common risk factors
- **Idiosyncratic returns**: asset-specific, diversifiable

$$
r_i = \alpha_i + \sum_{k=1}^{K} \beta_{ik} f_k + \epsilon_i
$$

where:
- $r_i$ = return of asset $i$
- $\beta_{ik}$ = exposure (loading) of asset $i$ to factor $k$
- $f_k$ = return of factor $k$
- $\epsilon_i$ = idiosyncratic return, $\epsilon_i \sim \mathcal{N}(0, \sigma_i^2)$

Portfolio return:
$$
r_p = \sum_i w_i r_i = \sum_{k} \left(\sum_i w_i \beta_{ik}\right) f_k + \sum_i w_i \epsilon_i = \sum_k B_{pk} f_k + \epsilon_p
$$

Portfolio variance:
$$
\sigma_p^2 = \mathbf{B}_p^\top \mathbf{F} \mathbf{B}_p + \mathbf{w}^\top \mathbf{\Delta} \mathbf{w}
$$

where $\mathbf{F}$ is the factor covariance matrix and $\mathbf{\Delta}$ is the diagonal idiosyncratic variance matrix.

### 1.2 Factor Selection

For a multi-asset portfolio spanning equities, rates, and geopolitical risks, we use three layers of factors:

**Macro factors** — drive cross-asset returns

| Factor | Proxy | Rationale |
|---|---|---|
| Oil price | Brent crude return | Key driver of inflation, EM, energy sector |
| Inflation surprise | 5Y breakeven rate change | Reprices bonds and rate-sensitive equities |
| Real rates | 5Y TIPS yield change | Core driver of bond and growth equity valuations |
| USD strength | DXY return | EM stress, commodity prices, risk-off |
| Credit spread | IG/HY OAS change | Risk appetite, funding conditions |
| Geopolitical risk | GPR index change | Tail risk, defense, safe havens |
| Food/agriculture | Bloomberg Agri index | Ukraine-specific transmission |

where: 
* **DXY**: US Dollar Index acroos basket of major currencies (EUR, JPY, GBP, CAD, SEK, CHF)
* **OAS**: Option Adjusted Spread
* **GPR**: Geopol risk index: uses newspaper text analysis

**Style factors** — drive cross-sectional equity returns

| Factor | Definition | Rationale |
|---|---|---|
| Momentum | 12M-1M price return | Trend following, crisis amplification |
| Value | B/P ratio | Mean reversion, cheap vs expensive |
| Quality | ROE, low leverage | Defensive in stress |
| Size | Market cap | Small cap more vulnerable in risk-off |
| Low volatility | Realized vol | Defensive factor |

**Sector/regional factors** — capture industry and geography

| Factor | Relevance |
|---|---|
| Energy | Gulf and Ukraine direct exposure |
| Defense | Ukraine specific |
| Tourism/hospitality | Gulf specific |
| EM Asia | Growth factor, oil importer stress |
| European equities | Ukraine proximity premium |
| Insurance | Rate sensitivity, cat risk |

### 1.3 Factor Covariance Matrix

In practice, $\mathbf{F}$ is estimated from historical returns using a shrinkage estimator to avoid overfitting:

$$
\hat{\mathbf{F}} = (1 - \delta) \mathbf{S} + \delta \mathbf{T}
$$

where $\mathbf{S}$ is the sample covariance, $\mathbf{T}$ is the shrinkage target (e.g. constant correlation), and $\delta$ is the shrinkage intensity estimated via Ledoit-Wolf.

Factor exposures $\beta_{ik}$ are estimated via time-series OLS for macro factors, and cross-sectional regression for style factors.

In [ ]:
from quant_risk.setup import base, macro

np, pd, plt = base()
fed_client, store, external_store = macro()

In [ ]:
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
if False:
    store.clear_cache()   
    external_store.clear_cache()

In [ ]:
fred_series = [
    "US10Y",
    "US2Y",
    "VIX",
    "IG_OAS",
    "HY_OAS",
    "BRENT",
    "BREAKEVEN5Y",
    "REAL5Y",
    "EURUSD",
    "DXY",
]

df_fred = store.build_panel(fred_series)
df_fred["Curve_Slope"] = df_fred["US10Y"] - df_fred["US2Y"]
print(df_fred.shape)
df_fred.isna().sum()

In [ ]:
ff = external_store.get_fama_french(frequency='daily')

style_factors = pd.DataFrame({
    'momentum': ff['Mom'],
    'value':    ff['HML'],
    'quality':  ff['RMW'],
    'size':     ff['SMB'],
    'low_vol':  -ff['Mkt-RF'],
}).dropna()

print(style_factors.tail(2))
print(style_factors.shape)

In [ ]:
yf_series = [
    "ENERGY_SECTOR",
    "DEFENSE_SECTOR",
    "EUROPE_EQ",
    "EM_ASIA",
    "INSURANCE",
    "TOURISM",
    "GLD",
    "TLT",
    "EEM",
    "IEF",
    "LQD",
    "HYG",
    "EWG",
    "EWI",
]

df_yf = external_store.build_panel(yf_series)
print(df_yf.shape)
df_yf.isna().sum()

In [ ]:
# GPR
gpr = external_store.get_gpr()
gpr.tail(2)

In [ ]:
# ── Factor assembly ───────────────────────────────────────────────────────────
# Convert all price series to returns and align to common index

# FRED -- rates and spreads are already levels, take diff for changes
# prices take pct_change
macro_factors = pd.DataFrame({
    'oil':            df_fred['BRENT'].pct_change(),
    'inflation_surp': df_fred['BREAKEVEN5Y'].diff(),
    'real_rates':     df_fred['REAL5Y'].diff(),
    'usd':            df_fred['DXY'].pct_change(),
    'geo_risk':       gpr.pct_change(),
    'curve_slope':    df_fred['Curve_Slope'].diff(),
    'vix_change':     df_fred['VIX'].diff(),
})

# Style factors -- already in return space from French library
style_factors = pd.DataFrame({
    'momentum': ff['Mom'],
    'value':    ff['HML'],
    'quality':  ff['RMW'],
    'size':     ff['SMB'],
    'low_vol':  -ff['Mkt-RF'],
})

# Sector/regional -- price ETFs to returns
sector_factors = pd.DataFrame({
    'energy_sector':  df_yf['ENERGY_SECTOR'].pct_change(),
    'defense_sector': df_yf['DEFENSE_SECTOR'].pct_change(),
    'europe_eq':      df_yf['EUROPE_EQ'].pct_change(),
    'em_asia':        df_yf['EM_ASIA'].pct_change(),
    'insurance':      df_yf['INSURANCE'].pct_change(),
    'tourism':        df_yf['TOURISM'].pct_change(),
})

# Safe havens -- useful for crisis analysis
safe_havens = pd.DataFrame({
    'gold':  df_yf['GLD'].pct_change(),
    'bonds': df_yf['TLT'].pct_change(),
})

# ── Align all to common index ─────────────────────────────────────────────────
factor_df = pd.concat(
    [macro_factors, style_factors, sector_factors, safe_havens],
    axis=1,
)

# drop to common non-null start -- 2008 where coverage is fullest
factor_df = factor_df['2008-01-01':]
factor_df = factor_df.dropna()

print(f"Factor panel shape: {factor_df.shape}")
print(f"Date range: {factor_df.index[0].date()} to {factor_df.index[-1].date()}")
print(f"Factors: {factor_df.columns.tolist()}")

# 1. simulate "AS of 01-05-2025"

In [ ]:
from sklearn.covariance import LedoitWolf

AS_OF_DATE = '2024-05-01'
factor_df_asof = factor_df[factor_df.index <= AS_OF_DATE].copy()

# Ledoit-Wolf on as-of data only
lw = LedoitWolf().fit(factor_df_asof)
F_cov_0824 = pd.DataFrame(lw.covariance_, index=factor_df_asof.columns, columns=factor_df_asof.columns)

vols_0824    = np.sqrt(np.diag(F_cov_0824.values))
F_corr_0824  = F_cov_0824 / np.outer(vols_0824, vols_0824)
F_corr_0824  = pd.DataFrame(F_corr_0824, index=factor_df_asof.columns, columns=factor_df_asof.columns)

print("Factor covariance matrix shape:", F_corr_0824.shape)
print("\nAnnualised vols as-of May 2024:")
print((pd.Series(vols_0824, index=factor_df_asof.columns) * np.sqrt(252)).round(3).to_string())

In [ ]:
def ewma_cov(returns: pd.DataFrame, lam: float = 0.94) -> pd.DataFrame:
    """
    Exponentially weighted covariance matrix.
    lambda=0.94 is the RiskMetrics standard for daily data.
    More recent observations get higher weight.
    """
    T, N = returns.shape
    weights = np.array([(1 - lam) * lam**t for t in range(T-1, -1, -1)])
    weights /= weights.sum()

    demeaned = returns.values - returns.mean().values
    cov = (demeaned * weights[:, None]).T @ demeaned

    return pd.DataFrame(cov, index=returns.columns, columns=returns.columns)

F_cov_ewma = ewma_cov(factor_df_asof, lam=0.94)
vols_ewma  = np.sqrt(np.diag(F_cov_ewma.values))
F_corr_ewma = F_cov_ewma / np.outer(vols_ewma, vols_ewma)
F_corr_ewma = pd.DataFrame(
    F_corr_ewma,
    index=factor_df_asof.columns,
    columns=factor_df_asof.columns,
)

print("Annualised vols (EWMA, as-of May 2024):")
print((pd.Series(vols_ewma, index=factor_df_asof.columns) * np.sqrt(252)).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 9))

heatmap_kwargs = dict(
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    annot_kws={'size': 10},
    linewidths=0.3,
)

sns.heatmap(F_corr_0824.round(2), ax=axes[0], **heatmap_kwargs)
axes[0].set_title('Factor correlation matrix 2008- MAY 1, 2024 (Ledoit-Wolf)', fontsize=12, pad=12)
axes[0].tick_params(axis='x', labelsize=13, rotation=90)
axes[0].tick_params(axis='y', labelsize=13, rotation=0)

sns.heatmap(F_corr_ewma.round(2), ax=axes[1], **heatmap_kwargs)
axes[1].set_title('Factor correlation matrix EWMA lambda=0.94 TO MAY 1, 2024 (recent regime)', fontsize=12, pad=12)
axes[1].tick_params(axis='x', labelsize=13, rotation=90)
axes[1].tick_params(axis='y', labelsize=13, rotation=0)

plt.tight_layout()
plt.show()

The EWMA matrix reveals a highly correlated recent regime. Factor diversification benefits are significantly lower than the long-run history suggests. Stress testing using the long-run matrix would underestimate portfolio risk in the current environment.

In [ ]:
def drop_correlated_factors(
    corr: pd.DataFrame,
    threshold: float = 0.70,
    must_keep: list[str] = None,
) -> list[str]:
    """
    Drop factors iteratively by choosing the most connected node first.
    Factors in must_keep are never dropped.
    """
    import collections

    must_keep = set(must_keep or [])
    remaining = list(corr.columns)
    to_drop = set()

    while True:
        counts = collections.Counter()
        for i in range(len(remaining)):
            for j in range(i+1, len(remaining)):
                if abs(corr.loc[remaining[i], remaining[j]]) > threshold:
                    counts[remaining[i]] += 1
                    counts[remaining[j]] += 1

        if not counts:
            break

        # find most connected factor that is not in must_keep
        for worst, count in counts.most_common():
            if worst not in must_keep:
                print(f"dropping {worst} (appears in {count} high-corr pairs)")
                to_drop.add(worst)
                remaining.remove(worst)
                break
        else:
            # all remaining high-corr factors are in must_keep, stop
            break

    keep = [c for c in corr.columns if c not in to_drop]
    return keep

keep_factors = drop_correlated_factors(F_corr_ewma, threshold=0.70, must_keep=['oil', 'vix_change'])
print(f"\nKeeping {len(keep_factors)} of {len(F_corr_ewma)} factors:")
print(keep_factors)


In [ ]:
# rebuild clean factor panel on as-of data only
factor_df_clean = factor_df_asof[keep_factors].copy()

# recompute EWMA covariance on clean factors
F_cov_clean = ewma_cov(factor_df_clean, lam=0.94)
vols_clean  = np.sqrt(np.diag(F_cov_clean.values))
F_corr_clean = F_cov_clean / np.outer(vols_clean, vols_clean)
F_corr_clean = pd.DataFrame(
    F_corr_clean,
    index=factor_df_clean.columns,
    columns=factor_df_clean.columns,
)

print(f"\nClean factor panel: {factor_df_clean.shape}")
print(f"Date range: {factor_df_clean.index[0].date()} to {factor_df_clean.index[-1].date()}")
print(f"Factors: {factor_df_clean.columns.tolist()}")
print(f"\nAnnualised vols (clean factors):")
print((pd.Series(vols_clean, index=factor_df_clean.columns) * np.sqrt(252)).round(3).to_string())

In [ ]:
# ── Portfolio asset data ──────────────────────────────────────────────────────
weights = pd.Series({
    "SP500":          0.20,
    "STOXX50":        0.10,
    "NIKKEI":         0.05,
    "EWG":            0.05,
    "GLD":            0.05,
    "TLT":            0.10,
    "IEF":            0.05,
    "LQD":            0.08,
    "HYG":            0.05,
    "ENERGY_SECTOR":  0.07,
    "DEFENSE_SECTOR": 0.05,
    "TOURISM":        0.05,
    "INSURANCE":      0.10,
})
weights = weights / weights.sum()  # normalise to 1

df_yf_assets = external_store.build_panel(portfolio_yf_series)
print(df_yf_assets.shape)
print(df_yf_assets.columns.tolist())

In [ ]:
# ── Asset returns ─────────────────────────────────────────────────────────────
asset_returns = df_yf_assets.pct_change()

# align to as-of date -- no look-ahead
asset_returns_asof = asset_returns[asset_returns.index <= AS_OF_DATE]

# align to clean factor panel
common_idx = factor_df_clean.index.intersection(asset_returns_asof.index)
X = factor_df_clean.loc[common_idx]
Y = asset_returns_asof.loc[common_idx]

print(f"Factor matrix X: {X.shape}")
print(f"Asset matrix Y:  {Y.shape}")
print(f"Date range: {common_idx[0].date()} to {common_idx[-1].date()}")
print(f"\nAsset NaN counts:")
print(Y.isna().sum())

In [ ]:
# ── OLS beta estimation ───────────────────────────────────────────────────────
from numpy.linalg import lstsq

X_const = np.column_stack([np.ones(len(X)), X.values])

betas  = {}
alphas = {}
r2s    = {}

for asset in Y.columns:
    y    = Y[asset].values
    mask = ~np.isnan(y)
    if mask.sum() < 100:
        continue

    coeffs, _, _, _ = lstsq(X_const[mask], y[mask], rcond=None)
    alphas[asset] = coeffs[0]
    betas[asset]  = coeffs[1:]

    y_hat  = X_const[mask] @ coeffs
    ss_res = ((y[mask] - y_hat) ** 2).sum()
    ss_tot = ((y[mask] - y[mask].mean()) ** 2).sum()
    r2s[asset] = 1 - ss_res / ss_tot

beta_df  = pd.DataFrame(betas,  index=factor_df_clean.columns).T
alpha_df = pd.Series(alphas, name='alpha')
r2_df    = pd.Series(r2s,    name='r2')

print("Beta matrix:")
print(beta_df.round(4).to_string())
print("\nR-squared per asset:")
print(r2_df.round(3).to_string())

In [ ]:
common_assets = [a for a in weights.index if a in beta_df.index]
w = weights[common_assets]
w = w / w.sum()

B_portfolio = beta_df.loc[common_assets].T @ w
B_portfolio.name = 'portfolio'

print("Portfolio factor betas:")
print(B_portfolio.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#d62728' if v < 0 else '#1f77b4' for v in B_portfolio.values]

ax.bar(B_portfolio.index, B_portfolio.values, color=colors, width=0.6)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Portfolio factor betas -- benign scenario (as-of May 2024)', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('beta')
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
print(factor_df_clean.columns.tolist())

---
## Part 2: Benign Portfolio Construction

### 2.1 Macro Scenario: Low Inflation, Growth, EM, Rate Cuts

The benign scenario is defined by:
- Inflation anchored at 2%, central banks cutting rates
- Global growth above trend, EM outperforming DM
- Oil stable to declining, USD soft
- Credit spreads tight, risk appetite high

**Desired factor exposures:**

|| Factor | Target exposure | Rationale |
|---|---|---|
| oil | Neutral to slight negative | Low commodity price thesis |
| real_rates | Negative | Long bonds, hurt by rising real rates |
| usd | Negative | Long EM, soft USD benefits |
| geo_risk | Negative | No geopolitical premium priced in |
| curve_slope | Positive | Steepening curve benefits banks and risk assets |
| vix_change | Negative | Risk-on, low volatility environment |
| momentum | Positive | Trend following in growth assets |
| value | Neutral | Not a value thesis |
| size | Positive | Small cap outperforms in risk-on |
| energy_sector | Neutral | Low oil, no energy overweight |
| defense_sector | Zero | No crisis scenario |
| insurance | Positive | Rate beneficiary |
| gold | Negative | Risk-on, no safe haven demand |
| bonds | Positive | Long duration, rate cut beneficiary |
### 2.2 Portfolio Legs and Asset-Factor Loadings

Each portfolio leg maps onto the factor model via its $\beta$ vector. Below are the assumed loadings based on the asset class characteristics.